# 📚 RAG Chatbot (LangChain + OpenAI)

This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline.

Flow at a glance:
1. Load structured coffee data (JSON)
2. Convert data into LangChain documents
3. Create embeddings and vector store
4. Retrieve relevant context
5. Answer questions using an LLM


In [ ]:
# Step 1: Import all required standard libraries and LangChain components
# ------------------------------------------------------------
import os
import json
import shutil

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


In [ ]:
# Step 2: Define configuration, file paths, and environment variables
# ------------------------------------------------------------

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

JSON_PATH = "../../data/coffee_data.json"

VECTOR_DB_DIRECTORY = "../../data/vector_store"
VECTOR_DB_COLLECTION = "coffee_documents"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


### Load JSON file

In [ ]:
# Step 3: Load JSON data and convert it into LangChain Document objects
# ------------------------------------------------------------
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw_items = json.load(f)

documents = []

for item in raw_items:
    product_url = item.get("product_url", "")
    item_text = item.get("item_main", "")

    content = f"""
Product URL:
{product_url}

Coffee Details:
{item_text}
""".strip()

    documents.append(
        Document(
            page_content=content,
            metadata={"source": product_url}
        )
    )

print(f"Loaded {len(documents)} coffee documents")


### Generate Embeddings

In [ ]:
# Step 4: Initialize OpenAI embedding model
# ------------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=0
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")


### Create local vectordb

In [ ]:
# Step 5: Create or reset the Chroma vector store on disk
# ------------------------------------------------------------
if os.path.exists(VECTOR_DB_DIRECTORY):
    shutil.rmtree(VECTOR_DB_DIRECTORY)

os.makedirs(VECTOR_DB_DIRECTORY, exist_ok=True)

print("Vector store directory reset")


### Load embedding model

In [ ]:
# Step 6: Add documents to the vector store and persist them
# ------------------------------------------------------------
import getpass
import os

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large"  # 3072 dimensions
)

print("OpenAI embedding model initialized")


## Retrieval 

In [ ]:
# Step 7: Create a retriever interface from the vector store
# ------------------------------------------------------------
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=VECTOR_DB_COLLECTION,
    persist_directory=VECTOR_DB_DIRECTORY,
)

print("Vector store created from JSON data")


In [ ]:
# Step 8: Initialize the OpenAI chat model (LLM)
# ------------------------------------------------------------
vector_store = Chroma(
    collection_name=VECTOR_DB_COLLECTION,
    persist_directory=VECTOR_DB_DIRECTORY,
    embedding_function=embedding_model,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 50})

print("✅ Vector store loaded for retrieval")


### Chatbot

In [ ]:
# Step 9: Define the system prompt that controls chatbot behavior
# ------------------------------------------------------------
chat_model = ChatOpenAI(
    model="gpt-4.1",
    temperature=0.3
)

print("Chat model initialized")


In [ ]:
# Step 10: Build the RAG chain combining retriever, prompt, and LLM
# ------------------------------------------------------------
prompt_template = ChatPromptTemplate.from_template(
    """
You are a helpful coffee assistant.You suggest coffee based on user preferences and cotext given to you.
Answer the question using the context below.
Use your knowledge to help user but don't invent any new information or add extra information which is not available in the cotext.
When listing items:
- Always return ALL matching coffees
- Do not give examples
- Do not summarize
- If 10 items match, list all 10
"

Context:
{context}

Question:
{question}

Answer:
"""
)

print("Prompt template ready")


In [ ]:

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

JSON_PATH = "../data/coffee_data.json"

VECTOR_DB_DIRECTORY = "../data/vector_store"
VECTOR_DB_COLLECTION = "coffee_documents"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


In [ ]:
# Step 11: Start an interactive command-line chat loop
# ------------------------------------------------------------
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | chat_model
    | StrOutputParser()
)

print("RAG chain constructed")


### Inferencing

In [ ]:
print("\n🤖 CoffeeBot ready. Type 'exit' to quit.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in {"exit", "quit"}:
        print("CoffeeBot: Goodbye!")
        break

    print(f"You: {user_input}\n")
    response = rag_chain.invoke(user_input)
    print(f"CoffeeBot: {response}\n")
    print("*************************************************\n")
